# BETH — Endpoint Kernel-Process Telemetry (eBPF)

| | |
|---|---|
| **Source** | Kaggle `katehighnam/beth-dataset` |
| **Origin** | Highnam et al., "BETH Dataset: Real Cybersecurity Data for Anomaly Detection Research" (ICML workshop 2021) |
| **License** | CC BY 4.0 |
| **Samples** | ~1.14M kernel process events (train/val/test splits, 8 hosts, honeypot compromise) |
| **Features** | processId, parentProcessId, userId, processName, eventName, args, returnValue, ... |
| **Labels** | `sus` (suspicious) and `evil` (confirmed malicious) |
| **Caveats** | `timestamp` = seconds since host boot → synthetic anchor applied (flagged). |
| **Production research suitability** | HIGH — real honeypot compromise, process-level EDR-like telemetry. |

In [1]:
import sys
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
from prep_utils import RAW, to_unified, dataset_report, numeric_summary, save_clean, save_unified_part

D = RAW / "cyber" / "beth"

## Load — official train/val/test splits

In [2]:
parts = []
for split in ["training", "validation", "testing"]:
    p = pd.read_csv(D / f"labelled_{split}_data.csv")
    p["split"] = split
    parts.append(p)
df = pd.concat(parts, ignore_index=True)
print(df.shape)
df.head(3)

(1141078, 17)


,timestamp,processId,threadId,parentProcessId,userId,mountNamespace,processName,hostName,eventId,eventName,stackAddresses,argsNum,returnValue,args,sus,evil,split
0,1809.495787,381,7337,1,100,4026532231,close,ip-10-100-1-120,157,prctl,"[140662171848350, 11649800180280676]",5,0,"[{'name': 'option', 'type': 'int', 'value': 'P...",1,0,training
1,1809.495832,381,7337,1,100,4026532231,close,ip-10-100-1-120,3,close,[140662171777451],1,0,"[{'name': 'fd', 'type': 'int', 'value': 19}]",1,0,training
2,1809.495921,381,7337,1,100,4026532231,close,ip-10-100-1-120,1010,sched_process_exit,[],0,0,[],1,0,training


## Cleaning

In [3]:
before = len(df)
df = df.drop_duplicates(subset=[c for c in df.columns if c != "split"]).reset_index(drop=True)
print(f"dropped {before - len(df)} duplicates")
print("missing:", {c: int(v) for c, v in df.isna().sum().items() if v})
df["processName"] = df["processName"].astype(str).str.strip()
df["eventName"] = df["eventName"].astype(str).str.strip().astype("category")
# BETH convention: userId >= 1000 → external/regular user, else system account
df["is_system_user"] = (df["userId"] < 1000)

dropped 0 duplicates
missing: {}


## Label verification
`evil` ⊆ malicious; `sus` = suspicious. Every evil row should also make sense
as at least suspicious per paper — check overlap, build final binary label.

In [4]:
print(pd.crosstab(df["sus"], df["evil"]))
df["label_bin"] = (df["evil"] == 1).astype("int8")
df["label_bin"].value_counts(normalize=True)

evil       0       1
sus                 
0     967564       0
1      15082  158432


label_bin
0    0.861156
1    0.138844
Name: proportion, dtype: float64

## Timestamp normalization
`timestamp` = seconds since boot per host. Anchor each host at the dataset's
capture epoch (May 2021) → synthetic but order-preserving per host.

In [5]:
anchor = pd.Timestamp("2021-05-01 00:00:00", tz="UTC")
df["event_time"] = anchor + pd.to_timedelta(df["timestamp"], unit="s")
df = df.sort_values("event_time").reset_index(drop=True)
df["event_time"].agg(["min", "max"])

min   2021-05-01 00:02:04.439221+00:00
max   2021-05-01 01:05:54.587643+00:00
Name: event_time, dtype: datetime64[ns, UTC]

## EDA

In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df["eventName"].value_counts().head(15).plot.barh(ax=axes[0], title="top syscall events")
df.groupby("split")["label_bin"].mean().plot.bar(ax=axes[1], title="evil rate by split")
df["processName"].value_counts().head(12).plot.barh(ax=axes[2], title="top processes")
plt.tight_layout(); plt.savefig("../reports/beth_eda.png", dpi=110); plt.show()

/tmp/ipykernel_176613/1491118754.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("../reports/beth_eda.png", dpi=110); plt.show()


In [7]:
numeric_summary(df, "beth")

,count,mean,std,min,25%,50%,75%,max
timestamp,1141078.0,1.367449e+03,1154.433376,1.244392e+02,4.612974e+02,9.033516e+02,2.327305e+03,3.954588e+03
processId,1141078.0,6.909070e+03,1816.699147,1.000000e+00,7.301000e+03,7.366000e+03,7.461000e+03,8.619000e+03
threadId,1141078.0,6.913038e+03,1807.393062,1.000000e+00,7.301000e+03,7.366000e+03,7.461000e+03,8.619000e+03
parentProcessId,1141078.0,2.467229e+03,2862.639715,0.000000e+00,1.870000e+02,1.385000e+03,4.489000e+03,7.672000e+03
userId,1141078.0,1.437311e+02,350.094691,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.001000e+03
mountNamespace,1141078.0,4.026532e+09,172.669668,4.026532e+09,4.026532e+09,4.026532e+09,4.026532e+09,4.026532e+09
eventId,1141078.0,2.372977e+02,354.831933,2.000000e+00,4.000000e+00,4.200000e+01,2.570000e+02,1.010000e+03
argsNum,1141078.0,2.671557e+00,1.250393,0.000000e+00,1.000000e+00,3.000000e+00,4.000000e+00,5.000000e+00
returnValue,1141078.0,3.018248e+00,322.346826,-1.150000e+02,0.000000e+00,0.000000e+00,0.000000e+00,3.276800e+04
sus,1141078.0,1.520615e-01,0.359081,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00


## Save clean + unified

In [8]:
keep = ["event_time", "processId", "parentProcessId", "userId", "processName",
        "eventName", "argsNum", "returnValue", "sus", "evil", "label_bin",
        "is_system_user", "split", "hostName" if "hostName" in df.columns else "mountNamespace"]
keep = [c for c in dict.fromkeys(keep) if c in df.columns]
clean = df[keep].copy()
save_clean(clean, "beth")
dataset_report(clean, "beth", label_col="label_bin",
               notes="timestamp = sec-since-boot anchored to 2021-05-01 UTC (synthetic, order-preserving).")

clean -> /home/suryaguru/StudioProjects/sentinel_fusion_ai/data/clean/beth.parquet (1,141,078 rows)


report -> /home/suryaguru/StudioProjects/sentinel_fusion_ai/reports/beth_stats.json


{'dataset': 'beth',
 'rows': 1141078,
 'columns': 14,
 'memory_mb': 134.6,
 'duplicate_rows': 3,
 'missing_by_column': {},
 'dtypes': {'event_time': 'datetime64[ns, UTC]',
  'processId': 'int64',
  'parentProcessId': 'int64',
  'userId': 'int64',
  'processName': 'str',
  'eventName': 'category',
  'argsNum': 'int64',
  'returnValue': 'int64',
  'sus': 'int64',
  'evil': 'int64',
  'label_bin': 'int8',
  'is_system_user': 'bool',
  'split': 'str',
  'hostName': 'str'},
 'notes': 'timestamp = sec-since-boot anchored to 2021-05-01 UTC (synthetic, order-preserving).',
 'label_distribution': {'0': 982646, '1': 158432},
 'imbalance_ratio': 6.2}

In [9]:
u = pd.DataFrame({
    "event_time": clean["event_time"],
    "event_subtype": clean["eventName"].astype(str),
    "user_id": clean["userId"].astype(str),
    "device_id": clean["hostName"].astype(str) if "hostName" in clean.columns else pd.NA,
    "severity": np.select([df["evil"] == 1, df["sus"] == 1], [4, 2], 1).astype("int8"),
    "label": clean["label_bin"],
    "time_is_synthetic": True,
})
attr_cols = ["processId", "parentProcessId", "processName", "argsNum", "returnValue", "sus", "split"]
u[attr_cols] = clean[attr_cols]
u = to_unified(u, source_dataset="beth", event_domain="cyber",
               event_type="process_exec", label_type="attack", attributes_cols=attr_cols)
save_unified_part(u, "beth")
u.head(3)

unified part -> /home/suryaguru/StudioProjects/sentinel_fusion_ai/data/unified/part_beth.parquet (1,141,078 rows)


,event_id,event_time,event_domain,event_type,event_subtype,source_dataset,user_id,device_id,src_ip,dst_ip,...,amount,duration_s,bytes_in,bytes_out,severity,label,label_type,attack_technique,time_is_synthetic,attributes
0,beth-0,2021-05-01 00:02:04.439221+00:00,cyber,process_exec,socket,beth,101,ip-10-100-1-129,<NA>,<NA>,...,NaN,NaN,NaN,NaN,1,0,attack,<NA>,True,"{""processId"":381,""parentProcessId"":1,""processN..."
1,beth-1,2021-05-01 00:02:04.439751+00:00,cyber,process_exec,socket,beth,100,ip-10-100-1-129,<NA>,<NA>,...,NaN,NaN,NaN,NaN,1,0,attack,<NA>,True,"{""processId"":378,""parentProcessId"":1,""processN..."
2,beth-2,2021-05-01 00:02:04.439958+00:00,cyber,process_exec,security_file_open,beth,0,ip-10-100-1-129,<NA>,<NA>,...,NaN,NaN,NaN,NaN,1,0,attack,<NA>,True,"{""processId"":1,""parentProcessId"":0,""processNam..."
